In [52]:
# Run this notebooks with chemgifs conda env

In [53]:
from rdkit import Chem
from collections import Counter
from pymol import cmd
from tqdm import tqdm
import numpy as np
import os
import pandas as pd
import pymol
import tarfile

In [54]:
# Define some paths
root = '../../../Documents_GPU/mtb-targeted-protein-degradation/scripts'
PATH_TO_DOCKING_RESULTS_ORIGINAL = os.path.join(root, "..", "processed", "unidock_docking", 'docking_results')
PATH_TO_DOCKING_RESULTS_REAL = os.path.join(root, "..", "processed", "unidock_REAL_docking", 'docking_results')
PATH_TO_INPUT_LIGANDS = os.path.join(root, "..", "processed", "unidock_REAL_docking", 'input_ligands')

# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))

In [4]:
DOCKING_RESULTS_ORIGINAL = {}
DOCKING_RESULTS_REAL = {}
DOCKING_RESULTS_REAL_BACKGROUND = {}

# For each pocket
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_ORIGINAL))):
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS_ORIGINAL, pocket, 'report.csv'), engine='python')
    DOCKING_RESULTS_ORIGINAL[pocket] = {i: j for i, j in zip(scores['compound'], scores['score'])}

# For each pocket
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_REAL))):
    # try:
    lines = open(os.path.join(PATH_TO_INPUT_LIGANDS, f"input_ligands_{pocket}.txt"), "r").readlines()
    lines = [i.strip().replace(".sdf", "").split("/")[-1] for i in lines]
    actives = set(lines[:100000])
    inactives = set(lines[100000:])
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'report.csv'))
    scores['set'] = ["inactive" if i in inactives else "active" for i in scores['compound']]
    scores_actives = scores[scores['set'] == 'active'].reset_index(drop=True)
    scores_inactives = scores[scores['set'] == 'inactive'].reset_index(drop=True)
    DOCKING_RESULTS_REAL[pocket] = {i: j for i, j in zip(scores_actives['compound'], scores_actives['score'])}
    DOCKING_RESULTS_REAL_BACKGROUND[pocket] = {i: j for i, j in zip(scores_inactives['compound'], scores_inactives['score'])}
    # except:
    #     pass

100%|██████████| 276/276 [00:32<00:00,  8.38it/s]


In [5]:
### SELECT TOP MOLECULES AND PREPARE GIF USING CHEMGIFS ###

In [6]:
ID_TO_SMILES = pd.read_csv(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL.tsv"), sep='\t')
ID_TO_SMILES = {i: j for i,j in zip(ID_TO_SMILES['id'], ID_TO_SMILES['smiles'])}

In [8]:
N = 10_000
ACTIVES = []
proteins = set(pocket_detection_data['Uniprot AC'])
ACTIVES_PER_PROTEIN = {i: set() for i in proteins}
for pocket in tqdm(sorted(DOCKING_RESULTS_REAL)):
    act = sorted(DOCKING_RESULTS_REAL[pocket], key = lambda x: DOCKING_RESULTS_REAL[pocket][x])[:N]
    ACTIVES.extend(act)
    ACTIVES_PER_PROTEIN[pocket.split("_")[1]].update(set(act))


# Get multi-target molecules
counts = Counter(ACTIVES)
counts_proteins = Counter([cpd for protein in proteins for cpd in ACTIVES_PER_PROTEIN[protein]])
active_21_proteins = [cmpd for cmpd, c in counts_proteins.items() if c >= 21]
print(f"TOP-{N} actives")
print(f"Compounds that are active at least once: {len(set(ACTIVES))}")
print(f"Compounds that are active in at least 21 proteins: {len(active_21_proteins)}")

# Get ID to SMILES mapping
SMILES = [ID_TO_SMILES[i] for i in active_21_proteins]

  0%|          | 0/276 [00:00<?, ?it/s]

100%|██████████| 276/276 [00:04<00:00, 65.32it/s]


TOP-10000 actives
Compounds that are active at least once: 620557
Compounds that are active in at least 21 proteins: 398


In [9]:
with open(os.path.join(root, "..", "processed", "unidock_REAL_docking", "multi_target_actives_smiles_TOP10k_21proteins.csv"), "w") as f:
    f.write("smiles\n")
    for smi in SMILES:
        f.write(smi.split()[0] + "\n")

In [11]:
N = 100
ACTIVES = []
proteins = set(pocket_detection_data['Uniprot AC'])
ACTIVES_PER_PROTEIN = {i: set() for i in proteins}
for pocket in tqdm(sorted(DOCKING_RESULTS_REAL)):
    act = sorted(DOCKING_RESULTS_REAL[pocket], key = lambda x: DOCKING_RESULTS_REAL[pocket][x])[:N]
    ACTIVES.extend(act)
    ACTIVES_PER_PROTEIN[pocket.split("_")[1]].update(set(act))


# Get multi-target molecules
counts = Counter(ACTIVES)
counts_proteins = Counter([cpd for protein in proteins for cpd in ACTIVES_PER_PROTEIN[protein]])
active_5_proteins = [cmpd for cmpd, c in counts_proteins.items() if c >= 5]
print(f"TOP-{N} actives")
print(f"Compounds that are active at least once: {len(set(ACTIVES))}")
print(f"Compounds that are active in at least 5 proteins: {len(active_5_proteins)}")

# Get ID to SMILES mapping
SMILES = [ID_TO_SMILES[i] for i in active_5_proteins]

100%|██████████| 276/276 [00:03<00:00, 89.91it/s] 

TOP-100 actives
Compounds that are active at least once: 15885
Compounds that are active in at least 5 proteins: 551


In [12]:
with open(os.path.join(root, "..", "processed", "unidock_REAL_docking", "multi_target_actives_smiles_TOP100_5proteins.csv"), "w") as f:
    f.write("smiles\n")
    for smi in SMILES:
        f.write(smi.split()[0] + "\n")

In [ ]:
### GET TOP POSES PER POCKET ###

In [27]:
for pocket in tqdm(sorted(DOCKING_RESULTS_REAL)):

    pocket = "alphafold3_P9WFW3_model_4_pocket_2"

    # Get top molecules
    top_mols = DOCKING_RESULTS_REAL[pocket]
    top_mols = sorted(top_mols, key = lambda x: top_mols[x])[:6]

    # Get path to structure
    st = "_".join(pocket.split("_")[:4])
    path_structure = os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, f'{st}.pdbqt')

    # Read tar file and extract top poses
    with tarfile.open(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'docking.tar.gz'), 'r:gz') as tar:

        # Create top poses directory
        os.makedirs(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'top_poses'), exist_ok=True)

        # Get top molecules
        for top_mol in top_mols:
            file = tar.extractfile(f"docking/{top_mol}_out.sdf").read()
            out_sdf = os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'top_poses', f"{top_mol}_out.sdf")
            with open(out_sdf, "wb") as f:
                f.write(file)

            # Create pdb file with st (pdbqt) and ligand (sdf) using pymol
            pymol.finish_launching(['pymol', '-cq'])
            cmd.reinitialize()
            cmd.load(path_structure, "prot")
            cmd.load(out_sdf, "lig")
            cmd.save(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket,
                                    'top_poses', f"{top_mol}_complex.pdb"), "prot lig")
            
            # Create PNG with interactions
            COMMAND = f"pandamap {os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket,'top_poses', f'{top_mol}_complex.pdb')} \
                --ligand UNK --output {os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket,'top_poses', f'{top_mol}_interactions.png')}"
            os.system(COMMAND)
            
    break

  0%|          | 0/276 [00:00<?, ?it/s]

Detecting interactions...
*** Using IMPROVED detection with stricter filtering ***
Before filtering: 1 total interactions
Calculating solvent accessibility...
Trying DSSP method first...
Error calculating solvent accessibility: 'HybridProtLigMapper' object has no attribute 'estimate_solvent_accessibility'
Falling back to realistic method
Using realistic solvent accessibility calculation...
Total protein atoms: 9074
Interacting residues to check: 6
Marking ('VAL', 241) as solvent accessible (score: 0.90)
Marking ('PRO', 530) as solvent accessible (score: 0.90)
Marking ('ASP', 242) as solvent accessible (score: 1.12)
Marking ('VAL', 491) as solvent accessible (score: 0.90)
Marking ('PHE', 240) as solvent accessible (score: 0.90)
Marking ('HIS', 531) as solvent accessible (score: 1.12)
Constraints: min=1, max=3 solvent accessible residues
Too many solvent-accessible residues detected (6), removing lowest scoring ones...
Removing ('VAL', 241) from solvent accessible (score: 0.90)
Removing 

  0%|          | 0/276 [00:41<?, ?it/s]

Interaction diagram saved to ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results/alphafold3_P9WFW3_model_4_pocket_2/top_poses/s_11____3203606____22135836_interactions.png
Analysis complete. Visualization saved to: ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results/alphafold3_P9WFW3_model_4_pocket_2/top_poses/s_11____3203606____22135836_interactions.png


In [ ]:
### SELECT TOP MOLECULES FOR VISUALIZATION ###

In [55]:
PATH_TO_DOCKING_RESULTS_REAL_LIGANDS = os.path.join(root, "..", "processed", "unidock_REAL_docking", 'docking_results_ligands')

N = 100
ACTIVES = []
proteins = set(pocket_detection_data['Uniprot AC'])
ACTIVES_PER_PROTEIN = {i: set() for i in proteins}
ACTIVES_PER_POCKET = {}
for pocket in tqdm(sorted(DOCKING_RESULTS_REAL)):
    act = sorted(DOCKING_RESULTS_REAL[pocket], key = lambda x: DOCKING_RESULTS_REAL[pocket][x])[:N]
    ACTIVES_PER_POCKET[pocket] = act
    ACTIVES.extend(act)
    ACTIVES_PER_PROTEIN[pocket.split("_")[1]].update(set(act))


counts = Counter(ACTIVES)
counts_proteins = Counter([cpd for protein in proteins for cpd in ACTIVES_PER_PROTEIN[protein]])
active_10_proteins = [cmpd for cmpd, c in counts_proteins.items() if c >= 10]

print(f"TOP-{N} actives")
print(f"Compounds that are active in at least 10 proteins: {len(active_10_proteins)}")

100%|██████████| 276/276 [00:03<00:00, 69.10it/s]

TOP-100 actives
Compounds that are active in at least 10 proteins: 89


In [67]:
mol_id = "s_22____21037368____24530012"

# Create poses directory
os.makedirs(os.path.join(PATH_TO_DOCKING_RESULTS_REAL_LIGANDS, mol_id), exist_ok=True)

In [70]:
for pocket in tqdm([i for i in sorted(DOCKING_RESULTS_REAL) if mol_id in ACTIVES_PER_POCKET[i]]):

    # Get path to structure
    st = "_".join(pocket.split("_")[:4])
    path_structure = os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, f'{st}.pdbqt')

    # Read tar file and extract top poses
    with tarfile.open(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'docking.tar.gz'), 'r:gz') as tar:

        # Extract file
        file = tar.extractfile(f"docking/{mol_id}_out.sdf").read()
        out_sdf = os.path.join(PATH_TO_DOCKING_RESULTS_REAL_LIGANDS, mol_id, f"{pocket}_{mol_id}_out.sdf")
        out_pdb = out_sdf.replace("_out.sdf", ".pdb")
        with open(out_sdf, "wb") as f:
            f.write(file)

        # Create pdb file with st (pdbqt) and ligand (sdf) using pymol
        pymol.finish_launching(['pymol', '-cq'])
        cmd.reinitialize()
        cmd.load(path_structure, "prot")
        cmd.load(out_sdf, "lig")
        cmd.save(out_pdb, "prot lig")

        # Remove sdf file
        os.remove(out_sdf)

        # Create PNG with interactions
        COMMAND = f"pandamap {out_pdb} \
            --ligand UNK --output {out_pdb.replace('.pdb', '.png')}"
        os.system(COMMAND)

  0%|          | 0/22 [00:00<?, ?it/s]

Detecting interactions...
*** Using IMPROVED detection with stricter filtering ***
Before filtering: 5 total interactions
Calculating solvent accessibility...
Trying DSSP method first...
Error calculating solvent accessibility: 'HybridProtLigMapper' object has no attribute 'estimate_solvent_accessibility'
Falling back to realistic method
Using realistic solvent accessibility calculation...
Total protein atoms: 13758
Interacting residues to check: 11
Marking ('HIS', 350) as solvent accessible (score: 1.12)
Marking ('GLY', 353) as solvent accessible (score: 1.00)
Marking ('TYR', 200) as solvent accessible (score: 1.12)
Marking ('THR', 227) as solvent accessible (score: 1.12)
Marking ('VAL', 198) as solvent accessible (score: 0.90)
Marking ('ASP', 288) as solvent accessible (score: 1.08)
Marking ('ARG', 228) as solvent accessible (score: 1.10)
Marking ('SER', 351) as solvent accessible (score: 1.12)
Marking ('VAL', 352) as solvent accessible (score: 0.90)
Marking ('PRO', 364) as solvent a

  5%|▍         | 1/22 [00:20<07:05, 20.28s/it]

Interaction diagram saved to ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results_ligands/s_22____21037368____24530012/alphafold2_P9WFS9_model_0_pocket_2_s_22____21037368____24530012.png
Analysis complete. Visualization saved to: ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results_ligands/s_22____21037368____24530012/alphafold2_P9WFS9_model_0_pocket_2_s_22____21037368____24530012.png
Detecting interactions...
*** Using IMPROVED detection with stricter filtering ***
Before filtering: 1 total interactions
Calculating solvent accessibility...
Trying DSSP method first...
Error calculating solvent accessibility: 'HybridProtLigMapper' object has no attribute 'estimate_solvent_accessibility'
Falling back to realistic method
Using realistic solvent accessibility calculation...
Total protein atoms: 13758
Interacting residues to check: 9
Marking ('GLY', 133) as solvent acces

  9%|▉         | 2/22 [00:40<06:41, 20.07s/it]

...

Interaction diagram saved to ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results_ligands/s_22____21037368____24530012/swissmodel_P9WFS9_model_1_pocket_3_s_22____21037368____24530012.png
Analysis complete. Visualization saved to: ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results_ligands/s_22____21037368____24530012/swissmodel_P9WFS9_model_1_pocket_3_s_22____21037368____24530012.png
Detecting interactions...
*** Using IMPROVED detection with stricter filtering ***
Before filtering: 9 total interactions
Calculating solvent accessibility...
Trying DSSP method first...
Error calculating solvent accessibility: 'HybridProtLigMapper' object has no attribute 'estimate_solvent_accessibility'
Falling back to realistic method
Using realistic solvent accessibility calculation...
Total protein atoms: 5191
Interacting residues to check: 11
Marking ('ILE', 112) as solvent acces

 95%|█████████▌| 21/22 [06:24<00:17, 17.17s/it]

Interaction diagram saved to ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results_ligands/s_22____21037368____24530012/swissmodel_P9WFU3_model_0_pocket_2_s_22____21037368____24530012.png
Analysis complete. Visualization saved to: ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results_ligands/s_22____21037368____24530012/swissmodel_P9WFU3_model_0_pocket_2_s_22____21037368____24530012.png
Detecting interactions...
*** Using IMPROVED detection with stricter filtering ***
Before filtering: 4 total interactions
Calculating solvent accessibility...
Trying DSSP method first...
Error calculating solvent accessibility: 'HybridProtLigMapper' object has no attribute 'estimate_solvent_accessibility'
Falling back to realistic method
Using realistic solvent accessibility calculation...
Total protein atoms: 7944
Interacting residues to check: 9
Marking ('GLY', 200) as solvent access

100%|██████████| 22/22 [06:41<00:00, 18.24s/it]

Interaction diagram saved to ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results_ligands/s_22____21037368____24530012/swissmodel_P9WFU5_model_0_pocket_2_s_22____21037368____24530012.png
Analysis complete. Visualization saved to: ../../../Documents_GPU/mtb-targeted-protein-degradation/scripts/../processed/unidock_REAL_docking/docking_results_ligands/s_22____21037368____24530012/swissmodel_P9WFU5_model_0_pocket_2_s_22____21037368____24530012.png
